# ********************************************************************************

# ***Air Tracker: Flight Analytics***

# ********************************************************************************

In [11]:
#importing libraries and giving the api key, headers frm RapidAPI - AreoDataBox

import time
import requests
import pandas as pd
import numpy as np
import os
import sqlite3
import json

api_key = '8d24e27489msh935f12948fbb869p1ec05bjsnc09fb805b2d9'
headers = {
	"x-rapidapi-key": f'{api_key}',
	"x-rapidapi-host": "aerodatabox.p.rapidapi.com",
	"Content-Type": "application/json"
}
#Defining a list of airports for my test
my_airports_iata_codes = ["MAA", "BLR", "HYD", "DEL", "CJB", "TRZ", "BOM", "IXM", "TRV", "COK"]

# 1. FETCHING AIRPORT DATA

In [4]:
# defining a function to fetch airport information for a given ICAO code
def fetch_airport_by_iata(iata_code):
    url = f"https://aerodatabox.p.rapidapi.com/airports/iata/{iata_code}"
    response = requests.get(url,headers=headers)
    time.sleep(2) # wait for 5 seconds before sending another request
    return response.json()

#defining a function to extract the required fields from the given json response
def extract_airport_info(airport_data):
    extracted_info = {
        'icao_code': airport_data.get('icao'),
        'iata_code': airport_data.get('iata'),
        'name': airport_data.get('fullName'),
        'city': airport_data.get('municipalityName'),
        'country': airport_data.get('country', {}).get('name'),
        'continent': airport_data.get('continent', {}).get('name'),
        'latitude': airport_data.get('location', {}).get('lat'),
        'longitude': airport_data.get('location', {}).get('lon'),
        'timezone': airport_data.get('timeZone')
    }
    return extracted_info

# testing the function
#Defining an array to store the airport details one by one
extracted_airports_data = []
for iata in my_airports_iata_codes:
    try:
        airport_data = fetch_airport_by_iata(iata)
        #Inserting all the airport data in to the array one by one
        extracted_airports_data.append(extract_airport_info(airport_data))
    except Exception as e:
        print(f"Error loading airport {iata}: {e}")
print(extracted_airports_data)

# Create a pandas DataFrame from the extracted data
airport_df = pd.DataFrame(extracted_airports_data)
display(airport_df)

[{'icao_code': 'VOMM', 'iata_code': 'MAA', 'name': 'Chennai', 'city': 'Chennai', 'country': 'India', 'continent': 'Asia', 'latitude': 12.990005, 'longitude': 80.1693, 'timezone': 'Asia/Kolkata'}, {'icao_code': 'VOBL', 'iata_code': 'BLR', 'name': 'Bangalore Bengaluru', 'city': 'Bangalore', 'country': 'India', 'continent': 'Asia', 'latitude': 13.197899, 'longitude': 77.7063, 'timezone': 'Asia/Kolkata'}, {'icao_code': 'VOHS', 'iata_code': 'HYD', 'name': 'Hyderabad Rajiv Gandhi', 'city': 'Hyderabad', 'country': 'India', 'continent': 'Asia', 'latitude': 17.231318, 'longitude': 78.429855, 'timezone': 'Asia/Kolkata'}, {'icao_code': 'VIDP', 'iata_code': 'DEL', 'name': 'New Delhi Indira Gandhi', 'city': 'New Delhi', 'country': 'India', 'continent': 'Asia', 'latitude': 28.5665, 'longitude': 77.1031, 'timezone': 'Asia/Kolkata'}, {'icao_code': 'VOCB', 'iata_code': 'CJB', 'name': 'Coimbatore', 'city': 'Coimbatore', 'country': 'India', 'continent': 'Asia', 'latitude': 11.029999, 'longitude': 77.0434

,icao_code,iata_code,name,city,country,continent,latitude,longitude,timezone
0,VOMM,MAA,Chennai,Chennai,India,Asia,12.990005,80.169300,Asia/Kolkata
1,VOBL,BLR,Bangalore Bengaluru,Bangalore,India,Asia,13.197899,77.706300,Asia/Kolkata
2,VOHS,HYD,Hyderabad Rajiv Gandhi,Hyderabad,India,Asia,17.231318,78.429855,Asia/Kolkata
3,VIDP,DEL,New Delhi Indira Gandhi,New Delhi,India,Asia,28.566500,77.103100,Asia/Kolkata
4,VOCB,CJB,Coimbatore,Coimbatore,India,Asia,11.029999,77.043400,Asia/Kolkata
5,VOTR,TRZ,Tiruchirappally Tiruchirapally Civil,Tiruchirappally,India,Asia,10.765399,78.709700,Asia/Kolkata
6,VABB,BOM,Mumbai Chhatrapati Shivaji,Mumbai,India,Asia,19.088700,72.867900,Asia/Kolkata
7,VOMD,IXM,Madurai,Madurai,India,Asia,9.834509,78.093400,Asia/Kolkata
8,VOTV,TRV,Trivandrum,Trivandrum,India,Asia,8.482119,76.920100,Asia/Kolkata
9,VOCI,COK,Kochi Cochin,Kochi,India,Asia,10.152000,76.401900,Asia/Kolkata


# 2. FETCHING FLIGHTS DATA


In [5]:
def fetch_flights_by_iata(iata_code):
    url = f"https://aerodatabox.p.rapidapi.com/flights/airports/iata/{iata_code}/2026-04-04T20:00/2026-04-05T08:00"
    response = requests.get(url,headers=headers)
    time.sleep(3) # wait for 3 seconds before sending another request
    return response.json()

flights_all = []

for iata in my_airports_iata_codes:
    try:
        flights_data = fetch_flights_by_iata(iata)
        flights_all.append(flights_data)
    except Exception as e:
        print(f"Error loading airport {iata}: {e}")

print(flights_all)

[{'departures': [{'movement': {'airport': {'icao': 'VOMD', 'iata': 'IXM', 'name': 'Madurai', 'countryCode': 'in', 'timeZone': 'Asia/Kolkata'}, 'scheduledTime': {'utc': '2026-04-04 14:30Z', 'local': '2026-04-04 20:00+05:30'}, 'terminal': '1', 'quality': ['Basic']}, 'number': '6E 7592', 'status': 'Unknown', 'codeshareStatus': 'Unknown', 'isCargo': False, 'aircraft': {'model': 'ATR 72'}, 'airline': {'name': 'IndiGo', 'iata': '6E', 'icao': 'IGO'}}, {'movement': {'airport': {'icao': 'VOCB', 'iata': 'CJB', 'name': 'Coimbatore', 'countryCode': 'in', 'timeZone': 'Asia/Kolkata'}, 'scheduledTime': {'utc': '2026-04-04 14:30Z', 'local': '2026-04-04 20:00+05:30'}, 'terminal': '1', 'quality': ['Basic']}, 'number': '6E 6812', 'status': 'Unknown', 'codeshareStatus': 'Unknown', 'isCargo': False, 'aircraft': {'model': 'Airbus A320'}, 'airline': {'name': 'IndiGo', 'iata': '6E', 'icao': 'IGO'}}, {'movement': {'airport': {'icao': 'VOHS', 'iata': 'HYD', 'name': 'Hyderabad', 'countryCode': 'in', 'timeZone': 

In [6]:
def extract_flight_info(flight_data, flight_type, origin_iata):
  data_to_return = []
  if flight_type == 'departure':
    departures = flight_data.get('departures', [])
    for departure in departures:
      scheduled_departure = departure.get('movement',{}).get('scheduledTime', {}).get('utc', '')
      flight_id = departure.get('number', '') + '_' + scheduled_departure
      flight_number = departure.get('number', '')
      aircraft_registration = departure.get('aircraft', {}).get('reg', '')
      destination_iata = departure.get('movement', {}).get('airport', {}).get('iata', '')
      flight_status = departure.get('status', '')
      actual_departure = departure.get('movement',{}).get('revisedTime', {}).get('utc', '')
      scheduled_arrival = ''
      actual_arrival = ''
      airline_code = departure.get('airline', {}).get('iata', '')
      data_to_return.append({
        'flight_id': flight_id,
        'flight_number': flight_number,
        'aircraft_registration': aircraft_registration,
        'origin_iata': origin_iata,
        'destination_iata': destination_iata,
        'scheduled_departure': scheduled_departure,
        'actual_departure': actual_departure,
        'scheduled_arrival': scheduled_arrival,
        'actual_arrival': actual_arrival,
        'status': flight_status,
        'airline_code': airline_code
      })
  elif flight_type == 'arrival':
    arrivals = flight_data.get('arrivals', [])
    for arrival in arrivals:
      flight_id = arrival.get('number', '') + '_' + arrival.get('movement',{}).get('scheduledTime', {}).get('utc', '')
      flight_number = arrival.get('number', '')
      aircraft_registration = arrival.get('aircraft', {}).get('reg', '')
      destination_iata = arrival.get('movement', {}).get('airport', {}).get('iata', '')
      scheduled_arrival = arrival.get('movement',{}).get('scheduledTime', {}).get('utc', '')
      flight_status = arrival.get('status', '')
      actual_arrival = arrival.get('movement',{}).get('revisedTime', {}).get('utc', '')
      scheduled_departure = ''
      actual_departure = ''
      airline_code = arrival.get('airline', {}).get('iata', '')
      data_to_return.append({
        'flight_id': flight_id,
        'flight_number': flight_number,
        'aircraft_registration': aircraft_registration,
        'origin_iata': origin_iata,
        'destination_iata': destination_iata,
        'scheduled_departure': scheduled_departure,
        'actual_departure': actual_departure,
        'scheduled_arrival': scheduled_arrival,
        'actual_arrival': actual_arrival,
        'status': flight_status,
        'airline_code': airline_code
      })
  return data_to_return

extracted_flights_info = []

for airport, flight in zip(my_airports_iata_codes, flights_all):
    try:
        flights_data = extract_flight_info(flight, 'departure', airport)
        #print(flights_data)
        extracted_flights_info.extend(flights_data)
        flights_data = extract_flight_info(flight, 'arrival', airport)
        #print(flights_data)
        extracted_flights_info.extend(flights_data)
    except Exception as e:
        print(f"Error loading airport {iata}: {e}")

print(extracted_flights_info)



[{'flight_id': '6E 7592_2026-04-04 14:30Z', 'flight_number': '6E 7592', 'aircraft_registration': '', 'origin_iata': 'MAA', 'destination_iata': 'IXM', 'scheduled_departure': '2026-04-04 14:30Z', 'actual_departure': '', 'scheduled_arrival': '', 'actual_arrival': '', 'status': 'Unknown', 'airline_code': '6E'}, {'flight_id': '6E 6812_2026-04-04 14:30Z', 'flight_number': '6E 6812', 'aircraft_registration': '', 'origin_iata': 'MAA', 'destination_iata': 'CJB', 'scheduled_departure': '2026-04-04 14:30Z', 'actual_departure': '', 'scheduled_arrival': '', 'actual_arrival': '', 'status': 'Unknown', 'airline_code': '6E'}, {'flight_id': '6E 193_2026-04-04 14:30Z', 'flight_number': '6E 193', 'aircraft_registration': '', 'origin_iata': 'MAA', 'destination_iata': 'HYD', 'scheduled_departure': '2026-04-04 14:30Z', 'actual_departure': '', 'scheduled_arrival': '', 'actual_arrival': '', 'status': 'Unknown', 'airline_code': '6E'}, {'flight_id': '6E 5149_2026-04-04 14:45Z', 'flight_number': '6E 5149', 'aircr

In [7]:
# Create a pandas DataFrame from the extracted data
extracted_flights_info_df = pd.DataFrame(extracted_flights_info)
display(extracted_flights_info_df)

,flight_id,flight_number,aircraft_registration,origin_iata,destination_iata,scheduled_departure,actual_departure,scheduled_arrival,actual_arrival,status,airline_code
0,6E 7592_2026-04-04 14:30Z,6E 7592,,MAA,IXM,2026-04-04 14:30Z,,,,Unknown,6E
1,6E 6812_2026-04-04 14:30Z,6E 6812,,MAA,CJB,2026-04-04 14:30Z,,,,Unknown,6E
2,6E 193_2026-04-04 14:30Z,6E 193,,MAA,HYD,2026-04-04 14:30Z,,,,Unknown,6E
3,6E 5149_2026-04-04 14:45Z,6E 5149,,MAA,BOM,2026-04-04 14:45Z,,,,Departed,6E
4,6E 7051_2026-04-04 14:50Z,6E 7051,,MAA,TRZ,2026-04-04 14:50Z,,,,Unknown,6E
...,...,...,...,...,...,...,...,...,...,...,...
1304,TR 758_2026-04-04 17:45Z,TR 758,9V-NCJ,TRZ,SIN,,,2026-04-04 17:45Z,,Expected,TR
1305,AK 29_2026-04-04 18:10Z,AK 29,9M-AGH,TRZ,KUL,,,2026-04-04 18:10Z,,Expected,AK
1306,IX 681_2026-04-04 21:00Z,IX 681,VT-AXW,TRZ,SIN,,,2026-04-04 21:00Z,,Expected,IX
1307,6E 1008_2026-04-04 22:50Z,6E 1008,VT-IXM,TRZ,SIN,,,2026-04-04 22:50Z,,Expected,6E


# 3. CALCULATING AND FETCHING AIRCRAFT DATA

In [8]:
# Extract aircraft_registration column, dropna, get unique items, convert tolist
unique_aircraft_registration_list = extracted_flights_info_df['aircraft_registration'].dropna().unique().tolist()
#filter empty values
unique_aircraft_registration_list = list(filter(None, unique_aircraft_registration_list))
# Display the unique flight numbers
print("Unique aircraft registration:", unique_aircraft_registration_list)

# Check the total count of unique flight numbers
print(f"\nTotal unique Aircrafts: {len(unique_aircraft_registration_list)}")



Unique aircraft registration: ['VT-NAC', 'VT-IMB', 'VT-IRW', 'VT-ICW', 'VT-BWK', 'VT-ISM', 'A6-EGD', 'VT-BWF', 'VT-IXI', 'VT-NHP', 'VT-NHD', '9V-SCG', 'VT-IZM', 'HS-ABV', 'VT-NHE', '9M-MXY', 'B-LAP', 'VT-IME', 'HS-TEX', 'VT-PPI', 'D-AIGX', 'VT-NCG', 'A9C-XA', 'VT-IPE', 'A7-BHQ', 'A6-ARA', 'A6-EPG', 'A6-LRA', 'A4O-OVG', 'VT-IIV', 'VT-NCD', 'A6-AUB', 'VT-SQD', 'G-ZBJC', 'VT-IYE', '9H-TJF', 'VT-IFM', 'VT-CIQ', 'VT-NCZ', 'VT-IKU', '9H-TJE', '9H-CXF', 'VT-IZF', 'VT-ILZ', 'VT-BXR', '4R-ABL', 'VT-ISS', 'VT-IPL', 'VT-IXE', 'VT-IPF', 'VT-SQA', 'VT-IBW', 'VT-IXT', '9H-TJC', 'VT-NCO', 'VT-IMU', 'A4O-MG', '9M-AGI', 'VT-BXT', 'VT-IBF', 'VT-YAM', 'VT-ISK', 'VT-TNE', 'OK-TVR', 'VT-YBA', 'VT-IZN', 'VT-NCL', 'VT-ILS', 'VT-CID', 'VT-ATE', 'VT-IMJ', 'VT-IVP', 'VT-NCV', 'VT-IFQ', 'VT-AII', 'VT-IJZ', 'A6-ECV', 'VT-GHD', 'VT-NCN', 'VT-IFS', 'VT-IFZ', 'VT-TQI', 'VT-IIM', 'VT-YAD', 'VT-ITI', 'VT-BWH', 'VT-CIH', 'VT-NHA', 'VT-IMW', 'A6-LRD', 'VT-IWT', 'VT-BKK', 'VT-RTU', 'VT-TNW', 'VT-YBC', 'VT-IBI', 'VT-IBL',

In [12]:
# Fetch aircraft data from RapidAPI AeroDataBox endpoint
def fetch_aircraft_by_registration(registration):
    url = f"https://aerodatabox.p.rapidapi.com/aircrafts/reg/{registration}"
    response = requests.get(url, headers=headers)
    time.sleep(1.5)  # Throttle requests to respect API rate limits
    return response.json()
aircraft_all = []
#first 10 items are fetched, remove this later.
for reg in unique_aircraft_registration_list:
    try:
        aircraft_data = fetch_aircraft_by_registration(reg)
        aircraft_all.append(aircraft_data)
    except requests.exceptions.RequestException as e:
        print(f"API error for aircraft {reg}: {e}")

    except ValueError as e:
        print(f"Invalid JSON response for aircraft {reg}: {e}")

    except Exception as e:
        print(f"Unexpected error loading aircraft {reg}: {e}")
print('All Aircrafts fetched from API: ',aircraft_all)

API error for aircraft D-AIGX: Expecting value: line 1 column 1 (char 0)
API error for aircraft VT-IIV: Expecting value: line 1 column 1 (char 0)
API error for aircraft VT-BXR: Expecting value: line 1 column 1 (char 0)
API error for aircraft VT-IPJ: Expecting value: line 1 column 1 (char 0)
API error for aircraft VT-BWM: Expecting value: line 1 column 1 (char 0)
API error for aircraft VT-IQI: Expecting value: line 1 column 1 (char 0)
API error for aircraft VT-NHC: Expecting value: line 1 column 1 (char 0)
API error for aircraft VT-ICK: Expecting value: line 1 column 1 (char 0)
All Aircrafts fetched from API:  [{'id': 14474, 'reg': 'VT-NAC', 'active': True, 'serial': '36299', 'hexIcao': '800742', 'airlineName': 'Air India', 'iataCodeShort': '788', 'icaoCode': 'B788', 'model': 'B788', 'modelCode': 'B787-8', 'numSeats': 256, 'rolloutDate': '2017-09-20', 'firstFlightDate': '2017-09-23', 'deliveryDate': '2017-10-11', 'registrationDate': '2017-10-11', 'typeName': 'Boeing 787-8', 'numEngines'

In [13]:
# Function to extract aircraft data matching the target table
def extract_aircraft_info(aircraft_data):
    extracted_info = {
        'registration': aircraft_data.get('reg'),
        'model': aircraft_data.get('model'),
        'manufacturer': aircraft_data.get('productionLine'),
        'icao_type_code': aircraft_data.get('icaoCode'),
        'owner': aircraft_data.get('airlineName')
    }
    return extracted_info

# TABLE OF AIRCRAFTS
extracted_aircrafts_data = []

for data in aircraft_all:
    extracted_aircrafts_data.append(extract_aircraft_info(data))

# Create a pandas DataFrame from the extracted data
aircraft_df = pd.DataFrame(extracted_aircrafts_data)
display(aircraft_df)

,registration,model,manufacturer,icao_type_code,owner
0,VT-NAC,B788,Boeing 787,B788,Air India
1,VT-IMB,A21N,Airbus A321 NEO,A21N,IndiGo
2,VT-IRW,A21N,Airbus A321 NEO,A21N,IndiGo
3,VT-ICW,A21N,Airbus A321 NEO,A21N,IndiGo
4,VT-BWK,B38M,Boeing 737 MAX,B38M,Air India Express
...,...,...,...,...,...
422,9V-NCJ,A21N,Airbus A321 NEO,A21N,Scoot
423,VT-AXW,B738,Boeing 737 NG,B738,Air India Express
424,9M-LRG,B38M,Boeing 737 MAX,B38M,Batik Air
425,9M-AGH,A20N,Airbus A320,A320,AirAsia


# 4. CALCULATING AIRPORT DELAYS

In [14]:
#Making a copy of extracted_flights_info_df data frame. Don't want to disturb the original.
copy_of_extracted_flights_info_df = extracted_flights_info_df.copy()

# Step 1: Parse all datetime columns (using utc=True handles ISO 8601 strings like '14:30Z')
time_cols = ['scheduled_departure', 'actual_departure', 'scheduled_arrival', 'actual_arrival']
for col in time_cols:
    copy_of_extracted_flights_info_df[col] = pd.to_datetime(
        copy_of_extracted_flights_info_df[col], errors='coerce', utc=True
    )
# Step 2: Fallback to arrival times if departure times are null/empty
copy_of_extracted_flights_info_df['scheduled_time'] = copy_of_extracted_flights_info_df[
    'scheduled_departure'
].fillna(copy_of_extracted_flights_info_df['scheduled_arrival'])

copy_of_extracted_flights_info_df['actual_time'] = copy_of_extracted_flights_info_df[
    'actual_departure'
].fillna(copy_of_extracted_flights_info_df['actual_arrival'])

# Step 3: Extract date & calculate delay minutes
copy_of_extracted_flights_info_df['delay_date'] = copy_of_extracted_flights_info_df[
    'scheduled_time'
].dt.strftime('%Y-%m-%d')

copy_of_extracted_flights_info_df['delay_min'] = (
    copy_of_extracted_flights_info_df['actual_time'] - copy_of_extracted_flights_info_df['scheduled_time']
).dt.total_seconds() / 60

# Step 4: Define delay and cancellation flags
copy_of_extracted_flights_info_df['is_delayed'] = copy_of_extracted_flights_info_df['delay_min'] > 0
copy_of_extracted_flights_info_df['is_canceled'] = (
    copy_of_extracted_flights_info_df['status'].str.lower() == 'canceled'
)

# Step 5: Group by airport and date to compute targeted metrics
airport_delay_df = copy_of_extracted_flights_info_df.groupby(['origin_iata', 'delay_date']).agg(
    total_flights=('flight_id', 'count'),
    delayed_flights=('is_delayed', 'sum'),
    avg_delay_min=('delay_min', lambda x: int(x[x > 0].mean()) if (x > 0).any() else 0),
    median_delay_min=('delay_min', lambda x: int(x[x > 0].median()) if (x > 0).any() else 0),
    canceled_flights=('is_canceled', 'sum')
).reset_index().rename(columns={'origin_iata': 'airport_iata'})


display(airport_delay_df)

,airport_iata,delay_date,total_flights,delayed_flights,avg_delay_min,median_delay_min,canceled_flights
0,BLR,2026-04-03,1,1,1450,1450,0
1,BLR,2026-04-04,231,100,37,25,7
2,BLR,2026-04-05,88,4,29,8,2
3,CJB,2026-04-04,17,0,0,0,0
4,CJB,2026-04-05,7,0,0,0,0
5,DEL,2026-04-04,429,0,0,0,9
6,DEL,2026-04-05,137,0,0,0,3
7,HYD,2026-04-04,148,0,0,0,0
8,HYD,2026-04-05,54,0,0,0,0
9,MAA,2026-04-04,137,0,0,0,2


# 5. CREATING SQL TABLE

In [16]:
# 1. Connect to SQLite (creates 'aviation.db' file in Colab workspace)
conn = sqlite3.connect('aviation_local.db')
cursor = conn.cursor()

# Enable Foreign Key support in SQLite
cursor.execute("PRAGMA foreign_keys = ON;")

# 2. Execute DDLs to create tables
cursor.executescript('''
DROP TABLE IF EXISTS airport;
DROP TABLE IF EXISTS aircraft;
DROP TABLE IF EXISTS flights;
DROP TABLE IF EXISTS airport_delays;

CREATE TABLE IF NOT EXISTS airport (
    airport_id INTEGER PRIMARY KEY AUTOINCREMENT,
    icao_code TEXT UNIQUE,
    iata_code TEXT UNIQUE,
    name TEXT,
    city TEXT,
    country TEXT,
    continent TEXT,
    latitude REAL,
    longitude REAL,
    timezone TEXT
);

CREATE TABLE IF NOT EXISTS aircraft (
    aircraft_id INTEGER PRIMARY KEY AUTOINCREMENT,
    registration TEXT UNIQUE,
    model TEXT,
    manufacturer TEXT,
    icao_type_code TEXT,
    owner TEXT
);

CREATE TABLE IF NOT EXISTS flights (
    flight_id TEXT,
    flight_number TEXT,
    aircraft_registration TEXT,
    origin_iata TEXT,
    destination_iata TEXT,
    scheduled_departure TEXT,
    actual_departure TEXT,
    scheduled_arrival TEXT,
    actual_arrival TEXT,
    status TEXT,
    airline_code TEXT
);

CREATE TABLE IF NOT EXISTS airport_delays (
    delay_id INTEGER PRIMARY KEY AUTOINCREMENT,
    airport_iata TEXT,
    delay_date TEXT,
    total_flights INTEGER,
    delayed_flights INTEGER,
    avg_delay_min INTEGER,
    median_delay_min INTEGER,
    canceled_flights INTEGER
);
''')

conn.commit()
print("Tables created successfully!")

Tables created successfully!


# 6. INSERTING DATA TO THE SQL TABLE

In [18]:
conn = sqlite3.connect('aviation_local.db')

# 1. Insert DataFrames into SQL tables
airport_df.to_sql('airport', conn, if_exists='append', index=False)
aircraft_df.to_sql('aircraft', conn, if_exists='append', index=False)
extracted_flights_info_df.to_sql('flights', conn, if_exists='append', index=False)
airport_delay_df.to_sql('airport_delays', conn, if_exists='append', index=False)

print("Data inserted successfully!")

Data inserted successfully!


# 7. SQL Queries

In [19]:
# Show the total number of flights for each aircraft model, listing the model and its count.

query_1 = """
SELECT
    a.model,
    COUNT(f.flight_id) AS total_flights
FROM aircraft a
LEFT JOIN flights f ON a.registration = f.aircraft_registration
GROUP BY a.model
ORDER BY total_flights DESC;
"""

df_1 = pd.read_sql_query(query_1, conn)
display(df_1)

,model,total_flights
0,A21N,219
1,A20N,177
2,B38M,112
3,A320,47
4,B738,41
5,AT75,35
6,B773,22
7,A333,21
8,B789,20
9,B788,19


In [20]:
# List all aircraft (registration, model) that have been assigned to more than 5 flights
query_2 = """
SELECT
    a.registration,
    a.model,
    COUNT(f.flight_id) AS flight_count
FROM aircraft a
JOIN flights f ON a.registration = f.aircraft_registration
GROUP BY a.registration, a.model
HAVING COUNT(f.flight_id) > 5
ORDER BY flight_count DESC;
"""

df_2 = pd.read_sql_query(query_2, conn)
display(df_2)

,registration,model,flight_count
0,VT-NCV,A21N,7
1,VT-IRW,A21N,6
2,VT-ISK,A20N,6
3,VT-NHA,A21N,6


In [21]:
# For each airport, display its name and the number of outbound flights, but only for airports with more than 5 flights.

query_3 = """
SELECT
    ap.name AS airport_name,
    COUNT(f.flight_id) AS outbound_flights
FROM airport ap
JOIN flights f ON ap.iata_code = f.origin_iata
GROUP BY ap.airport_id, ap.name
HAVING COUNT(f.flight_id) > 5
ORDER BY outbound_flights DESC;
"""

df_3 = pd.read_sql_query(query_3, conn)
display(df_3)

,airport_name,outbound_flights
0,New Delhi Indira Gandhi,566
1,Bangalore Bengaluru,320
2,Hyderabad Rajiv Gandhi,202
3,Chennai,178
4,Coimbatore,24
5,Tiruchirappally Tiruchirapally Civil,19


In [22]:
# Find the top 3 destination airports (name, city) by number of arriving flights, sorted by count descending.

query_4 = """
SELECT
    ap.name AS airport_name,
    ap.city,
    COUNT(f.flight_id) AS arriving_flights
FROM airport ap
JOIN flights f ON ap.iata_code = f.destination_iata
GROUP BY ap.airport_id, ap.name, ap.city
ORDER BY arriving_flights DESC
LIMIT 3;
"""

df_4 = pd.read_sql_query(query_4, conn)
display(df_4)

,airport_name,city,arriving_flights
0,Mumbai Chhatrapati Shivaji,Mumbai,117
1,New Delhi Indira Gandhi,New Delhi,92
2,Bangalore Bengaluru,Bangalore,68


In [23]:
# Show for each flight: number, origin, destination, and a label 'Domestic' or 'International' using CASE WHEN on country match.

query_5 = """
SELECT
    f.flight_number,
    f.origin_iata,
    f.destination_iata,
    orig.country AS origin_country,
    dest.country AS dest_country,
    CASE
        WHEN orig.country = dest.country THEN 'Domestic'
        ELSE 'International'
    END AS flight_type
FROM flights f
JOIN airport orig ON f.origin_iata = orig.iata_code
JOIN airport dest ON f.destination_iata = dest.iata_code;
"""

df_5 = pd.read_sql_query(query_5, conn)
display(df_5)

,flight_number,origin_iata,destination_iata,origin_country,dest_country,flight_type
0,6E 7592,MAA,IXM,India,India,Domestic
1,6E 6812,MAA,CJB,India,India,Domestic
2,6E 193,MAA,HYD,India,India,Domestic
3,6E 5149,MAA,BOM,India,India,Domestic
4,6E 7051,MAA,TRZ,India,India,Domestic
...,...,...,...,...,...,...
431,6E 7166,TRZ,BLR,India,India,Domestic
432,6E 7052,TRZ,MAA,India,India,Domestic
433,6E 2171,TRZ,HYD,India,India,Domestic
434,6E 7165,TRZ,BLR,India,India,Domestic


In [24]:
# Show the 5 most recent arrivals at “DEL” airport including flight number, aircraft, departure airport name, and arrival time, ordered by latest arrival.

query_6 = """
SELECT
    f.flight_number,
    f.aircraft_registration,
    orig.name AS departure_airport,
    COALESCE(f.actual_arrival, f.scheduled_arrival) AS arrival_time
FROM flights f
JOIN airport orig ON f.origin_iata = orig.iata_code
WHERE f.destination_iata = 'DEL'
ORDER BY arrival_time DESC
LIMIT 5;
"""

df_6 = pd.read_sql_query(query_6, conn)
display(df_6)

,flight_number,aircraft_registration,departure_airport,arrival_time
0,BZ 403,VT-BDO,Bangalore Bengaluru,2026-04-05 02:25Z
1,AI 2653,VT-TNQ,Bangalore Bengaluru,2026-04-05 01:52Z
2,6E 820,VT-NHA,Bangalore Bengaluru,2026-04-05 00:03Z
3,6E 6054,VT-NCV,Bangalore Bengaluru,2026-04-04 22:08Z
4,AI 2757,VT-TNI,Bangalore Bengaluru,2026-04-04 21:24Z


In [ ]:
# Find all airports with no arriving flights (never used as a destination in flights table).

query_7 = """
SELECT
    ap.iata_code,
    ap.name,
    ap.city,
    ap.country
FROM airport ap
LEFT JOIN flights f ON ap.iata_code = f.destination_iata
WHERE f.flight_id IS NULL;
"""

df_7 = pd.read_sql_query(query_7, conn)
display(df_7)

In [ ]:
#  For each airline, count the number of flights by status (e.g., 'On Time', 'Delayed', 'Cancelled') using CASE WHEN.

query_8 = """
SELECT
    f.airline_code,
    COUNT(f.flight_id) AS total_flights,
    SUM(CASE WHEN LOWER(f.status) LIKE '%on%time%' OR LOWER(f.status) = 'landed' THEN 1 ELSE 0 END) AS on_time_count,
    SUM(CASE WHEN LOWER(f.status) LIKE '%delay%' THEN 1 ELSE 0 END) AS delayed_count,
    SUM(CASE WHEN LOWER(f.status) LIKE '%cancel%' THEN 1 ELSE 0 END) AS cancelled_count
FROM flights f
GROUP BY f.airline_code;
"""

df_8 = pd.read_sql_query(query_8, conn)
display(df_8)

In [10]:
 # Show all cancelled flights, with aircraft and both airports, ordered by departure time descending.

query_9 = """
SELECT
    f.flight_number,
    f.scheduled_departure,
    f.aircraft_registration,
    orig.name AS origin_airport,
    dest.name AS destination_airport,
    f.status
FROM flights f
LEFT JOIN airport orig ON f.origin_iata = orig.iata_code
LEFT JOIN airport dest ON f.destination_iata = dest.iata_code
WHERE LOWER(f.status) LIKE '%cancel%'
ORDER BY f.scheduled_departure DESC;
"""

df_9 = pd.read_sql_query(query_9, conn)
display(df_9)

DatabaseError: Execution failed on sql '
SELECT
   f.flight_number,
   f.scheduled_departure,
   f.aircraft_registration,
   orig.name AS origin_airport,
   dest.name AS destination_airport,
   f.status
FROM flights f
LEFT JOIN airport orig ON f.origin_iata = orig.iata_code
LEFT JOIN airport dest ON f.destination_iata = dest.iata_code
WHERE LOWER(f.status) LIKE '%cancel%'
ORDER BY f.scheduled_departure DESC;
': no such table: flights

In [ ]:
# List all city pairs (origin-destination) that have more than 2 different aircraft models operating flights between them.

query_10 = """
SELECT
    orig.city AS origin_city,
    dest.city AS destination_city,
    COUNT(DISTINCT a.model) AS distinct_models_count
FROM flights f
JOIN airport orig ON f.origin_iata = orig.iata_code
JOIN airport dest ON f.destination_iata = dest.iata_code
JOIN aircraft a ON f.aircraft_registration = a.registration
GROUP BY orig.city, dest.city
HAVING COUNT(DISTINCT a.model) > 2
ORDER BY distinct_models_count DESC;
"""

df_10 = pd.read_sql_query(query_10, conn)
display(df_10)

In [ ]:
# For each destination airport, compute the % of delayed flights (status='Delayed') among all arrivals, sorted by highest percentage.

query_11 = """
SELECT
    ap.iata_code,
    ap.name AS destination_airport,
    COUNT(f.flight_id) AS total_arrivals,
    SUM(CASE WHEN LOWER(f.status) LIKE '%delay%' THEN 1 ELSE 0 END) AS delayed_arrivals,
    ROUND(
        (CAST(SUM(CASE WHEN LOWER(f.status) LIKE '%delay%' THEN 1 ELSE 0 END) AS REAL) / COUNT(f.flight_id)) * 100,
        2
    ) AS delay_percentage
FROM airport ap
JOIN flights f ON ap.iata_code = f.destination_iata
GROUP BY ap.iata_code, ap.name
ORDER BY delay_percentage DESC;
"""

df_11 = pd.read_sql_query(query_11, conn)
display(df_11)

# Gemini oda streamlit code

In [11]:
import streamlit as st
import sqlite3
import pandas as pd
import altair as alt

# -----------------------------------------------------------------------------
# 1. PAGE CONFIGURATION & DATABASE CONNECTION
# -----------------------------------------------------------------------------
st.set_page_config(
    page_title="Aviation Operations Analytics",
    page_icon="✈️",
    layout="wide",
    initial_sidebar_state="expanded"
)

DB_FILE = 'aviation.db'

@st.cache_resource
def get_db_connection():
    """Create a persistent connection to the SQLite database."""
    conn = sqlite3.connect(DB_FILE, check_same_thread=False)
    conn.execute("PRAGMA foreign_keys = ON;")
    return conn

conn = get_db_connection()

def run_query(sql_query, params=None):
    """Utility function to execute plain SQL and return a Pandas DataFrame."""
    try:
        return pd.read_sql_query(sql_query, conn, params=params)
    except Exception as e:
        st.error(f"Database Query Error: {e}")
        return pd.DataFrame()

# -----------------------------------------------------------------------------
# 2. SIDEBAR NAVIGATION & GLOBAL FILTERS
# -----------------------------------------------------------------------------
st.sidebar.title("✈️ Aviation Control")
page = st.sidebar.radio(
    "Navigate Modules",
    [
        "🏠 Homepage Dashboard",
        "🔍 Search & Filter Flights",
        "🏢 Airport Details Viewer",
        "⏱️ Delay Analytics",
        "🏆 Route Leaderboards"
    ]
)

st.sidebar.markdown("---")
st.sidebar.caption("Real-Time Analytics Connected to SQLite")


# -----------------------------------------------------------------------------
# MODULE 1: HOMEPAGE DASHBOARD
# -----------------------------------------------------------------------------
if page == "🏠 Homepage Dashboard":
    st.title("🏠 Aviation Operations Summary Dashboard")
    st.markdown("Overview of key operational metrics across airports, aircraft, and flights.")

    # Summary Statistics Queries
    total_airports = run_query("SELECT COUNT(*) as cnt FROM airport")['cnt'].iloc[0] if not run_query("SELECT COUNT(*) as cnt FROM airport").empty else 0
    total_flights = run_query("SELECT COUNT(*) as cnt FROM flights")['cnt'].iloc[0] if not run_query("SELECT COUNT(*) as cnt FROM flights").empty else 0
    avg_delay = run_query("SELECT ROUND(AVG(avg_delay_min), 1) as avg_d FROM airport_delays")['avg_d'].iloc[0] if not run_query("SELECT ROUND(AVG(avg_delay_min), 1) as avg_d FROM airport_delays").empty else 0
    cancelled_count = run_query("SELECT COUNT(*) as cnt FROM flights WHERE LOWER(status) LIKE '%cancel%'")['cnt'].iloc[0] if not run_query("SELECT COUNT(*) as cnt FROM flights WHERE LOWER(status) LIKE '%cancel%'").empty else 0

    # KPI Metric Cards
    col1, col2, col3, col4 = st.columns(4)
    col1.metric("Total Airports", f"{total_airports:,}")
    col2.metric("Total Flights", f"{total_flights:,}")
    col3.metric("System Avg Delay", f"{avg_delay} mins" if avg_delay else "0 mins")
    col4.metric("Cancelled Flights", f"{cancelled_count:,}")

    st.markdown("---")

    # Dashboard Grid
    c1, c2 = st.columns(2)

    with c1:
        st.subheader("✈️ Flights by Aircraft Model")
        df_model = run_query("""
            SELECT a.model, COUNT(f.flight_id) as flight_count
            FROM aircraft a
            LEFT JOIN flights f ON a.registration = f.aircraft_registration
            GROUP BY a.model ORDER BY flight_count DESC
        """)
        if not df_model.empty:
            chart_model = alt.Chart(df_model).mark_bar(color='#1f77b4').encode(
                x=alt.X('flight_count:Q', title='Total Flights'),
                y=alt.Y('model:N', sort='-x', title='Aircraft Model'),
                tooltip=['model', 'flight_count']
            ).properties(height=300)
            st.altair_chart(chart_model, use_container_width=True)
        else:
            st.info("No aircraft model data available.")

    with c2:
        st.subheader("📊 Flight Status Breakdown")
        df_status = run_query("""
            SELECT status, COUNT(*) as count
            FROM flights
            GROUP BY status
        """)
        if not df_status.empty:
            chart_status = alt.Chart(df_status).mark_arc(innerRadius=50).encode(
                theta='count:Q',
                color=alt.Color('status:N', legend=alt.Legend(title="Status")),
                tooltip=['status', 'count']
            ).properties(height=300)
            st.altair_chart(chart_status, use_container_width=True)
        else:
            st.info("No flight status data available.")


# -----------------------------------------------------------------------------
# MODULE 2: SEARCH & FILTER FLIGHTS
# -----------------------------------------------------------------------------
elif page == "🔍 Search & Filter Flights":
    st.title("🔍 Search & Filter Flights")

    # Load dynamic filter options
    airlines = ["All"] + run_query("SELECT DISTINCT airline_code FROM flights WHERE airline_code IS NOT NULL")['airline_code'].tolist()
    statuses = ["All"] + run_query("SELECT DISTINCT status FROM flights WHERE status IS NOT NULL")['status'].tolist()

    col_f1, col_f2, col_f3 = st.columns(3)
    with col_f1:
        search_query = st.text_input("Search Flight Number / Aircraft Tail", "")
    with col_f2:
        selected_airline = st.selectbox("Filter Airline", airlines)
    with col_f3:
        selected_status = st.selectbox("Filter Status", statuses)

    # Base SQL Query with Dynamic WHERE clauses
    sql_base = """
        SELECT
            f.flight_id,
            f.flight_number,
            f.airline_code,
            f.aircraft_registration,
            f.origin_iata,
            f.destination_iata,
            f.scheduled_departure,
            f.status,
            CASE
                WHEN orig.country = dest.country THEN 'Domestic'
                ELSE 'International'
            END AS category
        FROM flights f
        LEFT JOIN airport orig ON f.origin_iata = orig.iata_code
        LEFT JOIN airport dest ON f.destination_iata = dest.iata_code
        WHERE 1=1
    """
    params = []

    if search_query:
        sql_base += " AND (f.flight_number LIKE ? OR f.aircraft_registration LIKE ?)"
        params.extend([f"%{search_query}%", f"%{search_query}%"])
    if selected_airline != "All":
        sql_base += " AND f.airline_code = ?"
        params.append(selected_airline)
    if selected_status != "All":
        sql_base += " AND f.status = ?"
        params.append(selected_status)

    sql_base += " ORDER BY f.scheduled_departure DESC"

    df_filtered = run_query(sql_base, params)

    st.markdown(f"**Found {len(df_filtered)} matching flight records:**")
    st.dataframe(df_filtered, use_container_width=True)


# -----------------------------------------------------------------------------
# MODULE 3: AIRPORT DETAILS VIEWER
# -----------------------------------------------------------------------------
elif page == "🏢 Airport Details Viewer":
    st.title("🏢 Airport Details Viewer")

    df_airports = run_query("SELECT iata_code, name, city FROM airport ORDER BY name")

    if not df_airports.empty:
        airport_options = {f"{row['iata_code']} - {row['name']} ({row['city']})": row['iata_code'] for _, row in df_airports.iterrows()}
        selected_airport_label = st.selectbox("Select an Airport", list(airport_options.keys()))
        selected_iata = airport_options[selected_airport_label]

        # Query details
        airport_info = run_query("SELECT * FROM airport WHERE iata_code = ?", [selected_iata])

        if not airport_info.empty:
            info = airport_info.iloc[0]
            st.subheader(f"{info['name']} ({info['iata_code']})")

            # Display metadata table
            meta_col1, meta_col2, meta_col3 = st.columns(3)
            meta_col1.write(f"**City:** {info['city']}")
            meta_col1.write(f"**Country:** {info['country']}")
            meta_col2.write(f"**Latitude:** {info['latitude']}")
            meta_col2.write(f"**Longitude:** {info['longitude']}")
            meta_col3.write(f"**Timezone:** {info['timezone']}")
            meta_col3.write(f"**ICAO Code:** {info['icao_code']}")

            st.markdown("---")

            # Linked Flights Sub-tabs
            tab_out, tab_in = st.tabs(["🛫 Outbound Flights", "🛬 Inbound Flights"])

            with tab_out:
                df_out = run_query("""
                    SELECT flight_number, destination_iata, scheduled_departure, status, airline_code
                    FROM flights WHERE origin_iata = ? ORDER BY scheduled_departure DESC
                """, [selected_iata])
                st.dataframe(df_out, use_container_width=True)

            with tab_in:
                df_in = run_query("""
                    SELECT flight_number, origin_iata, scheduled_arrival, status, airline_code
                    FROM flights WHERE destination_iata = ? ORDER BY scheduled_arrival DESC
                """, [selected_iata])
                st.dataframe(df_in, use_container_width=True)
    else:
        st.warning("No airport data available in the database.")


# -----------------------------------------------------------------------------
# MODULE 4: DELAY ANALYTICS
# -----------------------------------------------------------------------------
elif page == "⏱️ Delay Analytics":
    st.title("⏱️ Airport Delay Analytics")

    st.subheader("Destination Delay Percentages")
    df_delay_pct = run_query("""
        SELECT
            ap.iata_code,
            ap.name AS destination_airport,
            COUNT(f.flight_id) AS total_arrivals,
            SUM(CASE WHEN LOWER(f.status) LIKE '%delay%' THEN 1 ELSE 0 END) AS delayed_arrivals,
            ROUND(
                (CAST(SUM(CASE WHEN LOWER(f.status) LIKE '%delay%' THEN 1 ELSE 0 END) AS REAL) / COUNT(f.flight_id)) * 100,
                2
            ) AS delay_percentage
        FROM airport ap
        JOIN flights f ON ap.iata_code = f.destination_iata
        GROUP BY ap.iata_code, ap.name
        ORDER BY delay_percentage DESC
    """)

    if not df_delay_pct.empty:
        col_table, col_chart = st.columns([1, 1])
        with col_table:
            st.dataframe(df_delay_pct, use_container_width=True)

        with col_chart:
            chart_delays = alt.Chart(df_delay_pct).mark_bar(color='#e74c3c').encode(
                x=alt.X('delay_percentage:Q', title='Delay %'),
                y=alt.Y('iata_code:N', sort='-x', title='Airport IATA'),
                tooltip=['destination_airport', 'delay_percentage', 'total_arrivals']
            ).properties(height=350)
            st.altair_chart(chart_delays, use_container_width=True)
    else:
        st.info("No delay records available to display.")


# -----------------------------------------------------------------------------
# MODULE 5: ROUTE LEADERBOARDS
# -----------------------------------------------------------------------------
elif page == "🏆 Route Leaderboards":
    st.title("🏆 Aviation Leaderboards & Route Analytics")

    tab_routes, tab_delayed_ap = st.tabs(["🔥 Busiest Routes", "⚠️ Most Delayed Airports"])

    with tab_routes:
        st.subheader("Top Busiest Origin-Destination Routes")
        df_routes = run_query("""
            SELECT
                f.origin_iata || ' ➔ ' || f.destination_iata AS route,
                orig.city AS origin_city,
                dest.city AS destination_city,
                COUNT(f.flight_id) as total_flights,
                COUNT(DISTINCT f.aircraft_registration) as unique_aircrafts
            FROM flights f
            JOIN airport orig ON f.origin_iata = orig.iata_code
            JOIN airport dest ON f.destination_iata = dest.iata_code
            GROUP BY f.origin_iata, f.destination_iata
            ORDER BY total_flights DESC
            LIMIT 10
        """)
        st.dataframe(df_routes, use_container_width=True)

    with tab_delayed_ap:
        st.subheader("Airports with Highest Average Delay (in Minutes)")
        df_top_delays = run_query("""
            SELECT
                a.iata_code,
                a.name,
                a.city,
                d.avg_delay_min,
                d.delayed_flights,
                d.canceled_flights
            FROM airport_delays d
            JOIN airport a ON d.airport_iata = a.iata_code
            ORDER BY d.avg_delay_min DESC
            LIMIT 10
        """)
        st.dataframe(df_top_delays, use_container_width=True)

2026-08-18 10:31:21.828 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-18 10:31:21.833 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-18 10:31:21.835 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-18 10:31:21.836 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-18 10:31:21.837 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-18 10:31:21.839 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-18 10:31:21.841 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-18 10:31:21.843 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [12]:
import streamlit as st
import sqlite3
import pandas as pd
import altair as alt


In [13]:
pip install streamlit

In [14]:
# 1. Install dependencies
!pip install streamlit pandas altair -q
!npm install -g localtunnel -q

# 2. Write the Streamlit app script to app.py
# (Run this after creating app.py using %%writefile app.py)

# 3. Get your IP address (used as the localtunnel password)
!curl https://loca.lt/mytunnelpassword

# 4. Run Streamlit in the background and expose port 8501
!streamlit run app.py & npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼
added 22 packages in 6s
⠼
⠼3 packages are looking for funding
⠼  run `npm fund` for details
⠼npm notice
npm notice New major version of npm available! 10.8.2 -> 12.0.2
npm notice Changelog: https://github.com/npm/cli/releases/tag/v12.0.2
npm notice To update run: npm install -g npm@12.0.2
npm notice
⠼34.75.105.2⠙⠹Usage: streamlit run [OPTIONS] [TARGET] [ARGS]...
Try 'streamlit run --help' for help.

Error: Invalid value: File does not exist: app.py
⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹your url is: https://soft-bars-bow.loca.lt
^C


# *************************************************************************************

# SAME BUT CHATGPT ODA ANSWER

# ************************************************************************************************************

In [15]:
import os
import sqlite3
from pathlib import Path

import pandas as pd
import plotly.express as px
import streamlit as st


# ============================================================
# CONFIGURATION
# ============================================================
st.set_page_config(
    page_title="Aviation Analytics Dashboard",
    page_icon="✈️",
    layout="wide",
)

DB_PATH = Path(__file__).resolve().parent / "aviation.db"



2026-08-18 10:32:46.213 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


NameError: name '__file__' is not defined

In [ ]:

# ============================================================
# DATABASE HELPERS
# ============================================================
def get_connection():
    """Create a fresh SQLite connection for each query."""
    if not DB_PATH.exists():
        raise FileNotFoundError(
            f"Database not found at {DB_PATH}. "
            "Run the data collection + SQL cells in the notebook first."
        )
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn


def query_db(sql, params=None):
    """Execute a read-only SQL query and return a DataFrame."""
    params = params or []
    with get_connection() as conn:
        return pd.read_sql_query(sql, conn, params=params)


def scalar_query(sql, params=None):
    """Execute a SQL query that returns one scalar value."""
    params = params or []
    with get_connection() as conn:
        row = conn.execute(sql, params).fetchone()
    return row[0] if row else 0


# ============================================================
# HEADER
# ============================================================
st.title("✈️ Aviation Analytics Dashboard")
st.caption("Interactive flight analysis powered by Streamlit + SQLite SQL queries")

if not DB_PATH.exists():
    st.error(
        "aviation.db was not found. Run the SQL creation and data insertion "
        "cells in the Colab notebook before launching this app."
    )
    st.stop()


# ============================================================
# SIDEBAR FILTERS
# ============================================================
st.sidebar.header("🔎 Filters")

status_options = query_db(
    "SELECT DISTINCT status FROM flights "
    "WHERE status IS NOT NULL AND TRIM(status) <> '' "
    "ORDER BY status"
)["status"].tolist()

origin_options = query_db(
    "SELECT DISTINCT origin_iata FROM flights "
    "WHERE origin_iata IS NOT NULL AND TRIM(origin_iata) <> '' "
    "ORDER BY origin_iata"
)["origin_iata"].tolist()

destination_options = query_db(
    "SELECT DISTINCT destination_iata FROM flights "
    "WHERE destination_iata IS NOT NULL AND TRIM(destination_iata) <> '' "
    "ORDER BY destination_iata"
)["destination_iata"].tolist()

airline_options = query_db(
    "SELECT DISTINCT airline_code FROM flights "
    "WHERE airline_code IS NOT NULL AND TRIM(airline_code) <> '' "
    "ORDER BY airline_code"
)["airline_code"].tolist()

date_bounds = query_db("""
    SELECT
        MIN(date(COALESCE(scheduled_departure, scheduled_arrival))) AS min_date,
        MAX(date(COALESCE(scheduled_departure, scheduled_arrival))) AS max_date
    FROM flights
""").iloc[0]

search_text = st.sidebar.text_input(
    "Flight number / airline",
    placeholder="e.g. AI101 or AI",
)

selected_status = st.sidebar.multiselect(
    "Flight status",
    options=status_options,
)

selected_origin = st.sidebar.multiselect(
    "Origin airport",
    options=origin_options,
)

selected_destination = st.sidebar.multiselect(
    "Destination airport",
    options=destination_options,
)

selected_airlines = st.sidebar.multiselect(
    "Airline",
    options=airline_options,
)

# Convert the database date strings into Python dates where possible.
import datetime as dt

min_date = pd.to_datetime(date_bounds["min_date"]).date()
max_date = pd.to_datetime(date_bounds["max_date"]).date()

selected_dates = st.sidebar.date_input(
    "Scheduled date range",
    value=(min_date, max_date),
    min_value=min_date,
    max_value=max_date,
)

if isinstance(selected_dates, tuple) and len(selected_dates) == 2:
    start_date, end_date = selected_dates
else:
    start_date = end_date = selected_dates


# ============================================================
# REUSABLE SQL FILTER BUILDER
# ============================================================
def build_flight_filters():
    clauses = ["1=1"]
    params = []

    if search_text.strip():
        clauses.append(
            "(LOWER(f.flight_number) LIKE LOWER(?) "
            "OR LOWER(f.airline_code) LIKE LOWER(?))"
        )
        pattern = f"%{search_text.strip()}%"
        params.extend([pattern, pattern])

    if selected_status:
        placeholders = ",".join(["?"] * len(selected_status))
        clauses.append(f"f.status IN ({placeholders})")
        params.extend(selected_status)

    if selected_origin:
        placeholders = ",".join(["?"] * len(selected_origin))
        clauses.append(f"f.origin_iata IN ({placeholders})")
        params.extend(selected_origin)

    if selected_destination:
        placeholders = ",".join(["?"] * len(selected_destination))
        clauses.append(f"f.destination_iata IN ({placeholders})")
        params.extend(selected_destination)

    if selected_airlines:
        placeholders = ",".join(["?"] * len(selected_airlines))
        clauses.append(f"f.airline_code IN ({placeholders})")
        params.extend(selected_airlines)

    clauses.append(
        "date(COALESCE(f.scheduled_departure, f.scheduled_arrival)) "
        "BETWEEN ? AND ?"
    )
    params.extend([str(start_date), str(end_date)])

    return " AND ".join(clauses), params


# ============================================================
# KPI CARDS
# ============================================================
kpi_col1, kpi_col2, kpi_col3 = st.columns(3)

total_airports = scalar_query("SELECT COUNT(*) FROM airport")

filter_sql, filter_params = build_flight_filters()

total_flights = scalar_query(
    f"SELECT COUNT(*) FROM flights f WHERE {filter_sql}",
    filter_params,
)

avg_delay = scalar_query(
    f"""
    SELECT COALESCE(ROUND(AVG(
        CASE
            WHEN actual_time IS NOT NULL AND scheduled_time IS NOT NULL
            THEN MAX((julianday(actual_time) - julianday(scheduled_time)) * 1440, 0)
        END
    ), 2), 0)
    FROM (
        SELECT
            CASE
                WHEN f.scheduled_departure IS NOT NULL
                     AND TRIM(f.scheduled_departure) <> ''
                THEN f.scheduled_departure
                ELSE f.scheduled_arrival
            END AS scheduled_time,
            CASE
                WHEN f.actual_departure IS NOT NULL
                     AND TRIM(f.actual_departure) <> ''
                THEN f.actual_departure
                ELSE f.actual_arrival
            END AS actual_time,
            f.*
        FROM flights f
    ) f
    WHERE {filter_sql}
    """,
    filter_params,
)

kpi_col1.metric("🏢 Total Airports", f"{total_airports:,}")
kpi_col2.metric("✈️ Flights Matching Filters", f"{int(total_flights):,}")
kpi_col3.metric("⏱️ Average Delay (min)", f"{float(avg_delay):.2f}")


# ============================================================
# MAIN TABS
# ============================================================
tab_dashboard, tab_flights, tab_airport, tab_delay, tab_routes = st.tabs(
    [
        "📊 Dashboard",
        "🔎 Flight Search",
        "🏢 Airport Details",
        "⏱️ Delay Analysis",
        "🏆 Route Leaderboards",
    ]
)


# ============================================================
# DASHBOARD TAB
# ============================================================
with tab_dashboard:
    st.subheader("Network Overview")

    status_df = query_db(
        f"""
        SELECT
            f.status,
            COUNT(*) AS total_flights
        FROM flights f
        WHERE {filter_sql}
        GROUP BY f.status
        ORDER BY total_flights DESC
        """,
        filter_params,
    )

    col1, col2 = st.columns(2)

    with col1:
        if not status_df.empty:
            fig = px.bar(
                status_df,
                x="status",
                y="total_flights",
                title="Flights by Status",
                text_auto=True,
            )
            st.plotly_chart(fig, use_container_width=True)

    with col2:
        route_df = query_db(
            f"""
            SELECT
                f.origin_iata || ' → ' || f.destination_iata AS route,
                COUNT(*) AS total_flights
            FROM flights f
            WHERE {filter_sql}
            GROUP BY f.origin_iata, f.destination_iata
            ORDER BY total_flights DESC
            LIMIT 10
            """,
            filter_params,
        )

        if not route_df.empty:
            fig = px.bar(
                route_df.sort_values("total_flights"),
                x="total_flights",
                y="route",
                orientation="h",
                title="Top 10 Routes",
                text_auto=True,
            )
            st.plotly_chart(fig, use_container_width=True)


# ============================================================
# FLIGHT SEARCH TAB
# ============================================================
with tab_flights:
    st.subheader("Search and Filter Flights")

    flights_df = query_db(
        f"""
        SELECT
            f.flight_number,
            f.airline_code,
            f.aircraft_registration,
            f.origin_iata,
            f.destination_iata,
            f.scheduled_departure,
            f.actual_departure,
            f.scheduled_arrival,
            f.actual_arrival,
            f.status
        FROM flights f
        WHERE {filter_sql}
        ORDER BY COALESCE(f.scheduled_departure, f.scheduled_arrival)
        LIMIT 1000
        """,
        filter_params,
    )

    st.write(f"Showing **{len(flights_df):,}** matching flights (maximum 1,000 displayed).")
    st.dataframe(flights_df, use_container_width=True, hide_index=True)


# ============================================================
# AIRPORT DETAILS TAB
# ============================================================
with tab_airport:
    st.subheader("Airport Details Viewer")

    airport_choices = query_db(
        """
        SELECT iata_code, name, city, country, timezone
        FROM airport
        WHERE iata_code IS NOT NULL
        ORDER BY iata_code
        """
    )

    if airport_choices.empty:
        st.warning("No airport records found.")
    else:
        selected_airport = st.selectbox(
            "Select an airport",
            airport_choices["iata_code"].tolist(),
        )

        airport = query_db(
            """
            SELECT
                iata_code,
                icao_code,
                name,
                city,
                country,
                continent,
                latitude,
                longitude,
                timezone
            FROM airport
            WHERE iata_code = ?
            """,
            [selected_airport],
        )

        if not airport.empty:
            st.dataframe(airport, use_container_width=True, hide_index=True)

        linked_flights = query_db(
            """
            SELECT
                flight_number,
                airline_code,
                origin_iata,
                destination_iata,
                scheduled_departure,
                scheduled_arrival,
                status
            FROM flights
            WHERE origin_iata = ? OR destination_iata = ?
            ORDER BY COALESCE(scheduled_departure, scheduled_arrival)
            LIMIT 500
            """,
            [selected_airport, selected_airport],
        )

        st.write(f"### Linked Flights ({len(linked_flights):,})")
        st.dataframe(linked_flights, use_container_width=True, hide_index=True)


# ============================================================
# DELAY ANALYSIS TAB
# ============================================================
with tab_delay:
    st.subheader("Airport Delay Analysis")

    delay_df = query_db(
        """
        SELECT
            airport_iata,
            SUM(total_flights) AS total_flights,
            SUM(delayed_flights) AS delayed_flights,
            ROUND(
                100.0 * SUM(delayed_flights) / NULLIF(SUM(total_flights), 0),
                2
            ) AS delay_percentage,
            ROUND(
                SUM(avg_delay_min * delayed_flights)
                / NULLIF(SUM(delayed_flights), 0),
                2
            ) AS weighted_avg_delay_min,
            SUM(canceled_flights) AS canceled_flights
        FROM airport_delays
        GROUP BY airport_iata
        ORDER BY delay_percentage DESC
        """
    )

    if not delay_df.empty:
        c1, c2 = st.columns(2)

        with c1:
            fig = px.bar(
                delay_df,
                x="airport_iata",
                y="delay_percentage",
                title="Delay Percentage by Airport",
                text_auto=True,
            )
            st.plotly_chart(fig, use_container_width=True)

        with c2:
            chart_df = delay_df.sort_values(
                "weighted_avg_delay_min", ascending=False
            ).head(10)

            fig = px.bar(
                chart_df.sort_values("weighted_avg_delay_min"),
                x="weighted_avg_delay_min",
                y="airport_iata",
                orientation="h",
                title="Average Delay by Airport",
                text_auto=True,
            )
            st.plotly_chart(fig, use_container_width=True)

        st.dataframe(delay_df, use_container_width=True, hide_index=True)


# ============================================================
# ROUTE LEADERBOARDS TAB
# ============================================================
with tab_routes:
    st.subheader("Route Leaderboards")

    busiest_routes = query_db(
        f"""
        SELECT
            f.origin_iata,
            f.destination_iata,
            f.origin_iata || ' → ' || f.destination_iata AS route,
            COUNT(*) AS total_flights
        FROM flights f
        WHERE {filter_sql}
        GROUP BY f.origin_iata, f.destination_iata
        ORDER BY total_flights DESC
        LIMIT 20
        """,
        filter_params,
    )

    most_delayed = query_db(
        """
        SELECT
            airport_iata,
            SUM(total_flights) AS total_flights,
            SUM(delayed_flights) AS delayed_flights,
            ROUND(
                100.0 * SUM(delayed_flights) / NULLIF(SUM(total_flights), 0),
                2
            ) AS delay_percentage,
            ROUND(
                SUM(avg_delay_min * delayed_flights)
                / NULLIF(SUM(delayed_flights), 0),
                2
            ) AS avg_delay_min
        FROM airport_delays
        GROUP BY airport_iata
        HAVING SUM(total_flights) > 0
        ORDER BY avg_delay_min DESC
        LIMIT 20
        """
    )

    left, right = st.columns(2)

    with left:
        st.markdown("### 🛫 Busiest Routes")
        st.dataframe(
            busiest_routes,
            use_container_width=True,
            hide_index=True,
        )

    with right:
        st.markdown("### ⚠️ Most Delayed Airports")
        st.dataframe(
            most_delayed,
            use_container_width=True,
            hide_index=True,
        )


# ============================================================
# FOOTER
# ============================================================
st.divider()
st.caption(
    "Data source: AeroDataBox API → SQLite database → Streamlit SQL queries. "
    "Dashboard queries are executed against the local SQL database on each interaction."
)